# Validation: Confirm PWMV +6pp on Qwen3.6-35B-A3B

Independent replication run for the headline claim of `caiovicentino1/qwen36-deepconf-probe`.

**Protocol**:
- Load Qwen3.6-35B-A3B + probe from our HF repo (not Colab-local)
- 30 Stage B prompts with **seed 777** (completely different from original seed 123)
- Run: greedy + PWMV (4 traces, temp 0.7) on each
- **Expected**: PWMV ≥ greedy + 4pp (confirms +6pp ±CI)

**Budget**: ~3.5h on RTX 6000. ~7 min per prompt (greedy ~3 min + PWMV ~5 min).

**Decision rule**:
- PWMV - greedy ≥ +4pp → **claim validated**, ship with confidence
- PWMV - greedy in [0, +3pp] → marginal, need larger N
- PWMV - greedy < 0 → CRITICAL: original result was seed-specific, retract

Target: confirm shippable product.

## Cell 1 — Setup

In [ ]:
import sys, subprocess, os, shutil
from pathlib import Path
def pip(*a): return subprocess.run([sys.executable, '-m', 'pip', *a], check=False)

try:
    import transformers
    from transformers.models.auto.configuration_auto import CONFIG_MAPPING_NAMES
    ok = 'qwen3_5_moe' in CONFIG_MAPPING_NAMES
except Exception: ok = False
if not ok:
    pip('install', '-q', 'accelerate', 'datasets', 'huggingface_hub==1.5.0',
        'safetensors', 'einops', 'scikit-learn', 'sentencepiece', 'tokenizers',
        'protobuf', 'scipy', 'hf_transfer')
    pip('uninstall', '-y', 'transformers', 'causal-conv1d')
    SRC = '/content/transformers_src'
    if Path(SRC).exists(): shutil.rmtree(SRC)
    subprocess.run(['git','clone','--quiet','--depth=1',
                    'https://github.com/huggingface/transformers.git', SRC], check=True)
    pip('install','--force-reinstall','--no-deps','--no-cache-dir', SRC)
    pip('install','--no-cache-dir','flash-linear-attention')
    for m in list(sys.modules):
        if m.startswith('transformers') or m.startswith('huggingface_hub'): del sys.modules[m]

try:
    from google.colab import userdata
    t = userdata.get('HF_TOKEN')
    if t:
        from huggingface_hub import login
        login(token=t); print('HF auth OK')
except: pass

import json, re, time, random, pickle
import numpy as np
import torch
import torch.nn.functional as F
from tqdm.auto import tqdm
from collections import defaultdict, Counter
from safetensors import safe_open
from huggingface_hub import snapshot_download, HfApi
os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '1'

OUT = Path('/content/validation'); OUT.mkdir(exist_ok=True)
MODEL_ID = 'Qwen/Qwen3.6-35B-A3B'
HF_PROBE = 'caiovicentino1/qwen36-deepconf-probe'
HF_STAGE_B = 'caiovicentino1/Qwen3.6-35B-A3B-mcr-stage-b'
HF_VALIDATION = 'caiovicentino1/qwen36-deepconf-probe'  # upload results back to main repo

PROBE_LAYER = 11
N_TRACES = 4
TEMP = 0.7
MAX_NEW = 3500
N_PROMPTS = 30
SEED = 777  # NEW seed (original was 123)
FORCE_SUFFIX = '\n</think>\n\nFinal answer: \\boxed{'

print('Validation setup: 30 prompts, seed 777, +6pp target')

## Cell 2 — Load Qwen3.6-35B-A3B + probe from HF

In [ ]:
from transformers import AutoTokenizer, AutoModelForImageTextToText

tok = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
if tok.pad_token_id is None: tok.pad_token = tok.eos_token

model = AutoModelForImageTextToText.from_pretrained(
    MODEL_ID, dtype=torch.bfloat16, device_map='cuda',
    attn_implementation='sdpa', trust_remote_code=True)
model.eval()
for p in model.parameters(): p.requires_grad_(False)
print(f'Model: {torch.cuda.memory_allocated()/1e9:.1f} GB')

# Download probe from HF (not local)
probe_dir = snapshot_download(HF_PROBE, repo_type='model', cache_dir='/content/cache',
                               allow_patterns=['probe_l11.pkl'])
with open(Path(probe_dir)/'probe_l11.pkl', 'rb') as f:
    probe = pickle.load(f)
print(f'Probe loaded from {HF_PROBE}')

# Install L11 hook
def get_layer_module(m, idx):
    cands = [m]
    if hasattr(m, 'model'): cands.append(m.model)
    for start in cands:
        for path in [('model','language_model','layers'),('language_model','layers'),('model','layers')]:
            cur = start; ok = True
            for p in path:
                if hasattr(cur, p): cur = getattr(cur, p)
                else: ok = False; break
            if ok and hasattr(cur, '__getitem__'):
                try: return cur[idx]
                except: continue
    raise RuntimeError(f'layer {idx} not found')

captured_l11 = {'chunks': []}
def l11_hook(m, i, o):
    h = o[0] if isinstance(o, tuple) else o
    captured_l11['chunks'].append(h[:, -1, :].detach().float().cpu())

layer_L = get_layer_module(model, PROBE_LAYER)
h_handle = layer_L.register_forward_hook(l11_hook)
print(f'L{PROBE_LAYER} hook installed')

## Cell 3 — Load 30 NEW prompts (seed 777, disjoint from original seed 123)

In [ ]:
corpus = snapshot_download(HF_STAGE_B, repo_type='dataset', cache_dir='/content/cache')
shards = sorted((Path(corpus)/'shards').glob('q*.safetensors'))

questions = []
seen = set()
for shard in shards:
    with safe_open(str(shard), framework='pt') as f:
        meta = f.metadata()
        q = meta['question']
        if q in seen: continue
        seen.add(q)
        opts = json.loads(meta['options'])
        if len(opts) != 10: continue
        questions.append(dict(question=q, options=opts, gold_letter=meta['gold_letter']))

# Reserve original seed 123 first 50 prompts (from n=50 canonical eval)
random.seed(123)
questions_copy_123 = questions.copy()
random.shuffle(questions_copy_123)
used_prompts = set(q['question'] for q in questions_copy_123[:50])

# Build fresh sample with seed 777, excluding already-used prompts
random.seed(SEED)
questions_new = [q for q in questions if q['question'] not in used_prompts]
random.shuffle(questions_new)
eval_set = questions_new[:N_PROMPTS]
print(f'{len(eval_set)} fresh prompts loaded (seed {SEED}, zero overlap with original n=50 eval)')

## Cell 4 — Eval helpers

In [ ]:
def extract_answer(text):
    post = text.split('</think>')[-1] if '</think>' in text else text
    for pattern in [
        r'\\boxed\{([A-J])\}',
        r'[Aa]nswer\s*(?:is\s*)?[:=]?\s*\*?\*?\(?([A-J])\)?\*?\*?',
        r'^\s*\(?([A-J])\)?\s*\.?\s*$',
    ]:
        m = re.search(pattern, post, re.MULTILINE)
        if m: return m.group(1).upper()
    m = re.findall(r'\\boxed\{([A-J])\}', text)
    if m: return m[-1]
    tail = post[-300:] if post else text[-300:]
    letters = re.findall(r'\b([A-J])\b', tail)
    return letters[-1] if letters else None

def fmt(q, opts):
    choices = '\n'.join(f'{chr(65+i)}. {o}' for i, o in enumerate(opts))
    content = (f'Answer the following multiple-choice question. '
               f'Reason step by step, then put the letter in \\boxed{{}}.\n\n'
               f'Question: {q}\n\nOptions:\n{choices}')
    return tok.apply_chat_template([{'role':'user','content':content}],
                                    tokenize=False, add_generation_prompt=True,
                                    enable_thinking=True)

def force_close(full_ids, prompt_len):
    gen = tok.decode(full_ids[prompt_len:].tolist(), skip_special_tokens=False)
    if '</think>' in gen:
        return tok.decode(full_ids[prompt_len:].tolist(), skip_special_tokens=True)
    full = tok.decode(full_ids.tolist(), skip_special_tokens=False) + FORCE_SUFFIX
    ctx = tok(full, return_tensors='pt', add_special_tokens=False).input_ids.cuda()
    with torch.no_grad():
        out = model.generate(ctx, max_new_tokens=15, do_sample=False,
                             pad_token_id=tok.pad_token_id, use_cache=True)
    suf = tok.decode(out[0, ctx.shape[1]:].tolist(), skip_special_tokens=True)
    return tok.decode(full_ids[prompt_len:].tolist(), skip_special_tokens=True) + FORCE_SUFFIX + suf

def gen_greedy(ids):
    captured_l11['chunks'] = []
    with torch.no_grad():
        out = model.generate(ids, max_new_tokens=MAX_NEW, do_sample=False,
                             pad_token_id=tok.pad_token_id, use_cache=True)
    return extract_answer(force_close(out[0], ids.shape[1]))

def gen_pwmv(ids):
    captured_l11['chunks'] = []
    with torch.no_grad():
        out = model.generate(ids, max_new_tokens=MAX_NEW, do_sample=True, temperature=TEMP,
                             num_return_sequences=N_TRACES, top_p=0.95,
                             pad_token_id=tok.pad_token_id, use_cache=True)
    chunks = captured_l11['chunks']
    if len(chunks) >= 2:
        embs = torch.stack(chunks[1:], dim=0).mean(dim=0).numpy()
        scores = probe.predict_proba(embs)[:, 1].tolist()
    else:
        scores = [1.0] * N_TRACES
    answers = [extract_answer(force_close(out[i], ids.shape[1])) for i in range(N_TRACES)]
    weighted = defaultdict(float)
    for a, s in zip(answers, scores):
        if a: weighted[a] += s
    return max(weighted, key=weighted.get) if weighted else None

print('Helpers ready')

## Cell 5 — Run validation (30 prompts × 2 methods, ~3.5h, crash-safe)

In [ ]:
results = {'greedy': [], 'pwmv': []}
t0 = time.time()

for qi, q in enumerate(tqdm(eval_set, desc='validate')):
    p = fmt(q['question'], q['options'])
    ids = tok(p, return_tensors='pt').input_ids.cuda()
    if ids.shape[1] > 1536: continue
    gold = q['gold_letter']
    try:
        g = gen_greedy(ids); results['greedy'].append((g, g == gold))
    except Exception as e:
        print(f'greedy err: {e}'); results['greedy'].append((None, False))
    try:
        p_pred = gen_pwmv(ids); results['pwmv'].append((p_pred, p_pred == gold))
    except torch.cuda.OutOfMemoryError:
        torch.cuda.empty_cache(); results['pwmv'].append((None, False))
    except Exception as e:
        print(f'pwmv err: {e}'); results['pwmv'].append((None, False))

    # Crash-safe checkpoint every 3
    if (qi+1) % 3 == 0:
        g_acc = np.mean([r[1] for r in results['greedy']])
        p_acc = np.mean([r[1] for r in results['pwmv']])
        delta = (p_acc - g_acc) * 100
        print(f'  [{qi+1}/{len(eval_set)}]  greedy={g_acc:.3f}  pwmv={p_acc:.3f}  Δ={delta:+.1f}pp')
        partial = {k: [(r[0], bool(r[1])) for r in v] for k, v in results.items()}
        with open(OUT / 'validation_partial.json', 'w') as f:
            json.dump(partial, f, indent=2)

h_handle.remove()

g_acc = np.mean([r[1] for r in results['greedy']])
p_acc = np.mean([r[1] for r in results['pwmv']])
delta_pp = (p_acc - g_acc) * 100

print(f'\n{'='*55}')
print(f'  VALIDATION RESULT (n={len(eval_set)}, seed={SEED}, {(time.time()-t0)/60:.0f} min)')
print(f'{'='*55}')
print(f'  greedy:  {g_acc:.3f}  ({sum(r[1] for r in results["greedy"])}/{len(results["greedy"])})')
print(f'  PWMV:    {p_acc:.3f}  ({sum(r[1] for r in results["pwmv"])}/{len(results["pwmv"])})')
print(f'  delta:   {delta_pp:+.1f}pp')
print(f'{'='*55}')

if delta_pp >= 4:
    print(f'  ✅ CLAIM VALIDATED: +{delta_pp:.1f}pp >= +4pp threshold')
    print(f'     Original n=50 seed=123: +6.0pp')
    print(f'     Replication n={len(eval_set)} seed={SEED}: +{delta_pp:.1f}pp')
elif delta_pp >= 0:
    print(f'  ⚠️  MARGINAL: +{delta_pp:.1f}pp. Replicates direction but under +4pp threshold.')
else:
    print(f'  ❌ FAILED: {delta_pp:+.1f}pp. Original result may have been seed-specific.')
print(f'{'='*55}')

## Cell 6 — Upload validation result to main repo

In [ ]:
validation_summary = {
    'validation_type': 'independent_seed_replication',
    'original_eval': {'seed': 123, 'n_prompts': 50, 'greedy': 0.700, 'pwmv': 0.760, 'delta_pp': 6.0},
    'replication_eval': {
        'seed': SEED,
        'n_prompts': len(eval_set),
        'greedy': float(g_acc),
        'pwmv': float(p_acc),
        'delta_pp': float(delta_pp),
    },
    'claim_validated': delta_pp >= 4,
    'verdict': 'VALIDATED' if delta_pp >= 4 else ('MARGINAL' if delta_pp >= 0 else 'FAILED'),
    'n_traces': N_TRACES,
    'temperature': TEMP,
    'max_new_tokens': MAX_NEW,
    'eval_date': time.strftime('%Y-%m-%d'),
}
with open(OUT / 'validation_n30_seed777.json', 'w') as f:
    json.dump(validation_summary, f, indent=2)

api = HfApi()
api.upload_file(path_or_fileobj=str(OUT/'validation_n30_seed777.json'),
                path_in_repo='validation_n30_seed777.json',
                repo_id=HF_VALIDATION, repo_type='model',
                commit_message=f'Independent validation seed=777 n={len(eval_set)}: delta={delta_pp:+.1f}pp')
api.upload_file(path_or_fileobj=str(OUT/'validation_partial.json'),
                path_in_repo='validation_n30_seed777_details.json',
                repo_id=HF_VALIDATION, repo_type='model',
                commit_message='Per-prompt validation details')
print(f'\n✅ Results uploaded: https://huggingface.co/{HF_VALIDATION}/blob/main/validation_n30_seed777.json')